# Исследование данных

In [1]:
import polars as pl

# Датасеты

In [2]:
train = pl.read_csv('data/train.csv', try_parse_dates=True)
test = pl.read_csv('data/test.csv', try_parse_dates=True)
events = pl.read_csv('data/events.csv.gz', try_parse_dates=True)

# Предпросмотр

In [3]:
train.head()

cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
str,datetime[μs],date,date,i64
"""ck_54a059eb7d3ea68b""",2025-11-21 09:30:41,2026-04-06,2026-04-07,0
"""ck_7e4de46eeab82974""",2025-09-23 10:10:24,2026-04-06,2026-04-07,0
"""ck_9320229ef6304522""",2026-03-04 00:08:02,2026-04-06,2026-04-07,0
"""ck_30ccd25bc1714ed9""",2026-04-05 10:52:40,2026-04-06,2026-04-07,0
"""ck_a77c5f05948cdeef""",2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [4]:
test.head()

cookie_id,cookie_created_at,window_start_ts,window_end_ts
str,datetime[μs],date,date
"""ck_315fb710a0e371e7""",2026-02-20 08:52:31,2026-04-20,2026-04-21
"""ck_a76ee3b3e3e522fd""",2026-01-19 13:42:15,2026-04-20,2026-04-21
"""ck_94c9a4d382689e82""",2026-04-19 06:28:37,2026-04-20,2026-04-21
"""ck_8eaf9509ad9462a0""",2026-01-19 17:18:06,2026-04-20,2026-04-21
"""ck_9a88a5a989cb5bc6""",2025-11-09 17:25:13,2026-04-20,2026-04-21


In [5]:
events.head()

cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
str,datetime[μs],i64,str,str,str,i64,str,str,str,str,i64,i64,i64
"""ck_5efbea1befdefe1b""",2026-04-26 09:11:24,200,"""item_view""","""desktop""","""Mozilla/5.0 (Macintosh; Intel …",1027450,"""elektronika""","""kaliningrad""","""pro""",null,null,null,null
"""ck_c4ca1434f3778f1d""",2026-04-20 14:04:31,100,"""search_results_view""","""web""","""Mozilla/5.0 (Windows NT 10.0; …",null,"""telefony""","""habarovsk""",null,"""iphone 13 128""",4,null,null
"""ck_d274382b19488771""",2026-04-13 12:02:56,200,"""item_view""","""WEB""","""Mozilla/5.0 (Windows NT 10.0; …",1048743,"""kvartiry_prodazha""","""kirov""","""private""",null,null,675,276
"""ck_fbbed2ff14944ce9""",2026-04-08 06:12:14,100,"""search_results_view""","""web""","""Mozilla/5.0 (Macintosh; Intel …",null,"""bytovaya_tehnika""","""novosibirsk""",null,"""пылесос dyson""",2,null,null
"""ck_56cc15c7c634cb9f""",2026-04-12 17:16:05,100,"""search_results_view""","""WEB""","""Mozilla/5.0 (Macintosh; Intel …",null,"""odezhda""","""sankt-peterburg""",null,"""костюм мужской""",2,null,null


# Дубликаты

In [6]:
train.is_duplicated().value_counts()

,count
bool,u32
false,11091


In [7]:
test.is_duplicated().value_counts()

,count
bool,u32
false,4909


In [8]:
events.is_duplicated().value_counts()

,count
bool,u32
false,319179
true,9726


In [9]:
print("Процент дубликатов в events:", round(events.is_duplicated().mean() * 100, 2))

Процент дубликатов в events: 2.96


### Вывод

В events почти 3% дубликатов. Перед дальнейшим исследованием их стоит удалить.

# Описание

In [10]:
train.describe()

statistic,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
str,str,str,str,str,f64
"""count""","""11091""","""11091""","""11091""","""11091""",11091.0
"""null_count""","""0""","""0""","""0""","""0""",0.0
"""mean""",null,"""2026-01-30 07:44:22.135966""","""2026-04-12 04:33:10.370570""","""2026-04-13 04:33:10.370570""",0.081057
"""std""",null,null,null,null,0.272934
"""min""","""ck_000c95f1408dcb00""","""2025-01-22 17:09:47""","""2026-04-06""","""2026-04-07""",0.0
"""25%""",null,"""2025-12-24 14:30:54""","""2026-04-09""","""2026-04-10""",0.0
"""50%""",null,"""2026-02-11 11:43:32""","""2026-04-12""","""2026-04-13""",0.0
"""75%""",null,"""2026-03-30 15:50:32""","""2026-04-15""","""2026-04-16""",0.0
"""max""","""ck_fffa4b3ab0203fcf""","""2026-04-18 22:45:04""","""2026-04-19""","""2026-04-20""",1.0


In [11]:
print("Процент ботов в train:", round(train.get_column('target').mean() * 100, 2))

Процент ботов в train: 8.11


In [12]:
test.describe()

statistic,cookie_id,cookie_created_at,window_start_ts,window_end_ts
str,str,str,str,str
"""count""","""4909""","""4909""","""4909""","""4909"""
"""null_count""","""0""","""0""","""0""","""0"""
"""mean""",null,"""2026-02-10 12:42:52.963740""","""2026-04-23 04:14:37.082909""","""2026-04-24 04:14:37.082909"""
"""std""",null,null,null,null
"""min""","""ck_00013fdfa0bd37dd""","""2025-01-15 16:57:05""","""2026-04-20""","""2026-04-21"""
"""25%""",null,"""2026-01-05 13:44:00""","""2026-04-21""","""2026-04-22"""
"""50%""",null,"""2026-02-23 11:11:24""","""2026-04-23""","""2026-04-24"""
"""75%""",null,"""2026-04-11 02:03:21""","""2026-04-25""","""2026-04-26"""
"""max""","""ck_fffa0622344a36ab""","""2026-04-25 23:00:54""","""2026-04-26""","""2026-04-27"""


In [13]:
events_unique = events.unique()

In [14]:
events_unique.describe()

statistic,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
str,str,str,f64,str,str,str,f64,str,str,str,str,f64,f64,f64
"""count""","""324042""","""324042""",324042.0,"""324042""","""324042""","""324042""",211104.0,"""291584""","""300574""","""192154""","""98929""",98929.0,106966.0,106966.0
"""null_count""","""0""","""0""",0.0,"""0""","""0""","""0""",112938.0,"""32458""","""23468""","""131888""","""225113""",225113.0,217076.0,217076.0
"""mean""",null,"""2026-04-15 20:42:22.213114""",212.13319,null,null,null,1.0300e6,null,null,null,null,2.902395,913.377101,535.585326
"""std""",null,null,139.436912,null,null,null,17285.349889,null,null,null,null,2.885921,538.416149,303.929918
"""min""","""ck_00013fdfa0bd37dd""","""2026-04-06 00:02:57""",100.0,"""captcha_shown""","""ANDROID""","""Avito/118.0 (Android 11; M2101…",1e6,"""avtomobili""","""astrahan""","""private""","""3к квартира""",1.0,0.0,0.0
"""25%""",null,"""2026-04-10 18:38:12""",100.0,null,null,null,1.014983e6,null,null,null,null,1.0,458.0,282.0
"""50%""",null,"""2026-04-15 03:39:50""",200.0,null,null,null,1.030038e6,null,null,null,null,2.0,884.0,533.0
"""75%""",null,"""2026-04-20 17:07:23""",210.0,null,null,null,1.044955e6,null,null,null,null,4.0,1358.0,789.0
"""max""","""ck_fffa4b3ab0203fcf""","""2026-04-26 23:59:50""",900.0,"""seller_page_view""","""web""","""python-urllib3/2.0.7""",1.059999e6,"""zhivotnye""","""yaroslavl""","""pro""","""электрик""",65.0,1920.0,1080.0


## Вывод

Нужно глубокое исследование events на предмет null значений, а также описательных статистик по столбцам.

# Исследование events

In [15]:
import plotly.express as px

In [16]:
def get_value_counts_with_percents(df: pl.DataFrame, column: str, desc: bool = False):
    
    return (
        df.get_column(column).fill_null("null") # Для дальнейшего отображения количество null в графиках
        .value_counts(name='count')
        .with_columns(
            (pl.col('count') / len(df) * 100).round(2).alias('percent')
        )
        .sort(by='count', descending=desc)
    )

In [17]:
def plot_value_counts_with_percents(df: pl.DataFrame, column: str, width = 1000, height = 800):
    
    plot_df = get_value_counts_with_percents(df, column)
    
    fig = px.bar(
        plot_df,
        x=column,
        y='count',
        text='percent'
    )
    
    fig.update_traces(
        texttemplate='%{text:.2f}%'
    )
    
    fig.update_layout(width=width, height=height)
    
    fig.show()

## Пропуски по столбцам

In [18]:
def plot_null_counts_with_percents(df: pl.DataFrame):
    
    null_df = (
        df.null_count()
        .transpose(include_header=True, header_name='column', column_names=['null_count'])
        .with_columns(
            (pl.col('null_count') / len(df) * 100).round(2).alias('percent')
        )
        .sort(by='null_count')
    )
    
    fig = px.bar(
        null_df,
        x='column',
        y='null_count',
        text='percent'
    )
    
    fig.update_traces(
        texttemplate='%{text:.2f}%'
    )
    
    fig.show()

In [19]:
plot_null_counts_with_percents(events_unique)

In [20]:
def get_column_nulls_distribution(events: pl.DataFrame) -> pl.DataFrame:
    df = events.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    
    null_counts = (
        df.group_by('target')
        .agg([
            pl.col(col).is_null().sum().alias(col)
            for col in events.columns
        ])
    )
    
    group_sizes = df.group_by('target').agg(pl.len().alias('group_size'))
    
    null_counts = null_counts.join(group_sizes, on='target')
    
    long = null_counts.unpivot(
        index=['target', 'group_size'],
        on=events.columns,
        variable_name='column',
        value_name='null_count'
    )
    
    long = long.with_columns(
        (pl.col('null_count') / pl.col('group_size') * 100).round(2).alias('null_percent'),
    )
    
    wide = (
        long
        .pivot(on='target', index='column', values=['null_count', 'null_percent'])
        .fill_null(0)
        .rename({
            'null_count_0': 'human_null_count',
            'null_count_1': 'bot_null_count',
            'null_percent_0': 'human_null_percent',
            'null_percent_1': 'bot_null_percent',
        })
        .with_columns(
            (pl.col('bot_null_percent') - pl.col('human_null_percent')).alias('diff_bot_human_percent')
        )
    )
    
    return wide

In [21]:
column_null_target_distribution = get_column_nulls_distribution(events_unique)

In [22]:
fig = px.bar(
    column_null_target_distribution,
    x='column',
    y='diff_bot_human_percent',
    text='diff_bot_human_percent'
)

fig.update_layout(barmode='relative')

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.show()

## Количество event на cookie

In [103]:
event_per_cookie = (
    events_unique.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    .group_by('cookie_id', 'target')
    .agg(pl.len().alias('event_count'))
    .group_by('target')
    .agg(
        pl.col('event_count').mean().alias('mean_event_count'),
        pl.col('event_count').median().alias('median_event_count'),
        pl.col('event_count').std().alias('std_event_count'),
    )
    .with_columns(
        (pl.col('std_event_count') / pl.col('mean_event_count')).alias('cv_event_count')
    )
)

event_per_cookie

target,mean_event_count,median_event_count,std_event_count,cv_event_count
i64,f64,f64,f64,f64
0,18.668465,14.0,16.38353,0.877605
1,50.527253,44.0,29.032278,0.574587


Количество событий у ботов значительно выше, чем в обычных пользователей. Коэфициент вариации говорит о том, что относительная вариативность у ботов выше, чем у обычных пользователей.

## event_ts

In [ ]:
ts_dist = (
    events_unique.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    .group_by('target')
    .agg(pl.col('event_ts').dt.hour().is_between(0, 6).mean().alias('night_activity'))
)

ts_dist

target,night_activity
i64,f64
0,0.250325
1,0.559418


Боты чаще активнее ночью.

In [81]:
per_user_entropy = (
    events_unique.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    .group_by(['cookie_id', 'target'])
    .agg(pl.col('event_ts').dt.hour().value_counts().struct.field('count').entropy(base=2).alias('user_entropy'))
    .group_by('target')
    .agg(pl.col('user_entropy').mean())
)

per_user_entropy

target,user_entropy
i64,f64
1,2.008191
0,1.394358


Боты ведут себя менее регулярно по времени, без чёткого личного расписания активности. 

## Исследование корреляции между количеством собитий и энтропией по часам

In [119]:
user_features = (
    events_unique
    .group_by('cookie_id')
    .agg(
        pl.len().alias('event_count'),
        pl.col('event_ts').dt.hour().value_counts().struct.field('count').entropy(base=2).alias('user_entropy')
    )
)

user_features.select(
    pl.corr('event_count', 'user_entropy').alias('pearson_corr')
)

pearson_corr
f64
0.609688


In [120]:
user_features.select(
    pl.corr('event_count', 'user_entropy', method='spearman').alias('spearman_corr')
)

spearman_corr
f64
0.693371


Умеренная кореляция, стоит проверить корреляцию объёма собитый с соотношением энтропии с количеством собитый.

In [121]:
user_features = user_features.with_columns(
    (pl.col('user_entropy') / pl.col('event_count').clip(2).log(2)).alias('entropy_ratio')
)

In [122]:
user_features.select(
    pl.corr('event_count', 'entropy_ratio').alias('pearson_corr_ratio'),
    pl.corr('event_count', 'entropy_ratio', method='spearman').alias('spearman_corr_ratio')
)

pearson_corr_ratio,spearman_corr_ratio
f64,f64
0.194666,0.250822


In [127]:
(
    user_features.
    join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    .group_by('target').
    agg(
        pl.col('entropy_ratio').mean().alias('mean_entropy_ratio'),
        pl.col('entropy_ratio').median().alias('median_entropy_ratio'),
        pl.col('entropy_ratio').std().alias('std_entropy_ratio')
    )
)

target,mean_entropy_ratio,median_entropy_ratio,std_entropy_ratio
i64,f64,f64,f64
0,0.355063,0.383028,0.192167
1,0.367168,0.379897,0.138073


После удаления эффекта объёма разница по entropy_ratio между 
людьми и ботами практически исчезла (по mean и median), значит 
признак сам по себе нерелевантен. Лучше остановиться на сырых 
user_entropy и event_count.

## event_name

In [23]:
plot_value_counts_with_percents(events_unique, 'event_name')

In [24]:
def get_event_names_distribution_by_target(events: pl.DataFrame) -> pl.DataFrame:
    
    df: pl.DataFrame = events.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    
    dist = (
        df.group_by(['event_name', 'target'])
        .agg(pl.len().cast(pl.Int64).alias('count'))
        .with_columns(
            (pl.col('count') / pl.col('count').sum().over('target') * 100).round(2).alias('percent'),
        )
        .sort(by=['target', 'count'])
    )
    
    wide = (
        dist
        .pivot(on="target", index="event_name", values=["count", "percent"])
        .fill_null(0)
    )
    
    wide = wide.rename({
        'count_0': 'count_human',
        'count_1': 'count_bot',
        'percent_0': 'percent_human',
        'percent_1': 'percent_bot'
    })
    
    wide = wide.with_columns(
        (pl.col("count_bot") - pl.col("count_human")).cast(pl.Int64).alias("diff_bot_human_count"),
        (pl.col("percent_bot") - pl.col("percent_human")).round(2).alias("diff_bot_human_percent")
    )

    return wide

In [25]:
event_names_target_distribution = get_event_names_distribution_by_target(events_unique)

In [26]:
fig = px.bar(
    event_names_target_distribution,
    x='event_name',
    y='diff_bot_human_percent',
    text='diff_bot_human_percent'
)

fig.update_layout(barmode='relative')

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.show()

Ботам чаще, чем людям, показывается каптча. Можно сделать признак.

## platform

In [27]:
plot_value_counts_with_percents(events_unique, 'platform')

Необходимо привести платформы к единообразному виду, так как сейчас происходит дублирование информации.

In [28]:
platform_norm = events_unique.with_columns(
    pl.col('platform').str.to_lowercase()
)

In [29]:
plot_value_counts_with_percents(platform_norm, 'platform')

In [30]:
def get_platform_distribution_by_target(events: pl.DataFrame) -> pl.DataFrame:
    
    df: pl.DataFrame = events.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    
    dist = (
        df.group_by(['platform', 'target'])
        .agg(pl.len().cast(pl.Int64).alias('count'))
        .with_columns(
            (pl.col('count') / pl.col('count').sum().over('target') * 100).round(2).alias('percent')
        )
        .sort(by=['target', 'count'])
    )
    
    wide = (
        dist
        .pivot(on="target", index="platform", values=["count", "percent"])
        .fill_null(0)
    )
    
    wide = wide.rename({
        'count_0': 'count_human',
        'count_1': 'count_bot',
        'percent_0': 'percent_human',
        'percent_1': 'percent_bot'
    })
    
    wide = wide.with_columns(
        (pl.col("count_bot") - pl.col("count_human")).cast(pl.Int64).alias("diff_bot_human_count"),
        (pl.col("percent_bot") - pl.col("percent_human")).round(2).alias("diff_bot_human_percent")
    )

    return wide

In [31]:
platform_target_distribution = get_platform_distribution_by_target(platform_norm)

In [32]:
fig = px.bar(
    platform_target_distribution,
    x='platform',
    y='diff_bot_human_percent',
    text='diff_bot_human_percent'
)

fig.update_layout(barmode='relative')

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.show()

Боты чаще, чем люди, используют web и desktop. Desktop, возможно, cli инструменты.

## item_category

In [33]:
plot_value_counts_with_percents(events_unique, 'item_category', width=1500)

In [34]:
def get_item_distribution_by_target(events: pl.DataFrame) -> pl.DataFrame:
    
    df: pl.DataFrame = events.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    
    dist: pl.DataFrame = (
        df.group_by(['item_category', 'target'])
        .agg(pl.len().cast(pl.Int64).alias('count'))
        .with_columns(
            (pl.col('count') / pl.col('count').sum().over('target') * 100).round(2).alias('percent'),
            (pl.col('item_category').fill_null('null').alias('item_category'))
        )
        .sort(by=['target', 'count'])
    )
    
    wide: pl.DataFrame = (
        dist
        .pivot(on="target", index="item_category", values=["count", "percent"])
        .fill_null(0)
    )
    
    wide = wide.rename({
        'count_0': 'count_human',
        'count_1': 'count_bot',
        'percent_0': 'percent_human',
        'percent_1': 'percent_bot'
    })
    
    wide = wide.with_columns(
        (pl.col("count_bot") - pl.col("count_human")).cast(pl.Int64).alias("diff_bot_human_count"),
        (pl.col("percent_bot") - pl.col("percent_human")).round(2).alias("diff_bot_human_percent")
    )

    return wide

In [35]:
item_target_distribution = get_item_distribution_by_target(events_unique)

In [36]:
fig = px.bar(
    item_target_distribution,
    x='item_category',
    y='diff_bot_human_percent',
    text='diff_bot_human_percent',
)

fig.update_layout(barmode='relative')

fig.show()

### Вывод

У ботов чаще незаполнен item_category, возможно, потому что они не парсят сам товар, а лишь страницы поиска.

## item_location

In [37]:
plot_value_counts_with_percents(events_unique, 'item_location', width=1500)

## seller_type

In [38]:
plot_value_counts_with_percents(events_unique, 'seller_type')

## Временные метки

### Временные дельты между действиями cookie

In [39]:
def add_event_deltas(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.sort(['cookie_id', 'event_ts'])
        .with_columns(
            pl.col('event_ts')
            .diff()
            .dt.total_seconds()
            .over('cookie_id')
            .alias('delta_sec')
        )
    )

In [40]:
events_deltas = add_event_deltas(events_unique)
events_deltas.head()

cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,delta_sec
str,datetime[μs],i64,str,str,str,i64,str,str,str,str,i64,i64,i64,i64
"""ck_00013fdfa0bd37dd""",2026-04-24 18:41:16,210,"""photo_swipe""","""ANDROID""","""Mozilla/5.0 (Linux; Android 12…",1012349,"""rabota""","""moskva""","""private""",null,null,null,null,null
"""ck_00013fdfa0bd37dd""",2026-04-24 19:16:53,200,"""item_view""","""android""","""Mozilla/5.0 (Linux; Android 12…",1054355,"""rabota""","""moskva""","""private""",null,null,null,null,2137
"""ck_00013fdfa0bd37dd""",2026-04-24 19:16:54,210,"""photo_swipe""","""ANDROID""","""Mozilla/5.0 (Linux; Android 12…",1015683,"""detskie_tovary""","""habarovsk""","""pro""",null,null,null,null,1
"""ck_00013fdfa0bd37dd""",2026-04-24 19:17:05,200,"""item_view""","""android""","""Mozilla/5.0 (Linux; Android 12…",1028622,"""avtomobili""","""moskva""","""private""",null,null,null,null,11
"""ck_00013fdfa0bd37dd""",2026-04-24 20:05:05,500,"""login""","""android""","""Mozilla/5.0 (Linux; Android 12…",null,null,null,null,null,null,null,null,2880


In [41]:
events_deltas.get_column('delta_sec').drop_nulls().describe()

statistic,value
str,f64
"""count""",308042.0
"""null_count""",0.0
"""mean""",1190.369161
"""std""",5191.580312
"""min""",0.0
"""25%""",16.0
"""50%""",41.0
"""75%""",118.0
"""max""",93013.0


In [42]:
events_deltas.get_column('delta_sec').drop_nulls().median()

41.0

### Квантили delta_sec

In [43]:
def delta_quantiles(events: pl.DataFrame) -> pl.DataFrame:
    quantiles = [0.25, 0.5, 0.7, 0.8, 0.9, 0.95, 0.97, 0.99, 0.995, 0.999]
    
    deltas = events.select('delta_sec').drop_nulls()
    
    return deltas.select(
        (pl.quantile('delta_sec', q)).alias(f'{(q)}') for q in quantiles
    )

In [44]:
delta_quantiles(events_deltas)

0.25,0.5,0.7,0.8,0.9,0.95,0.97,0.99,0.995,0.999
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
16.0,41.0,90.0,174.0,3594.0,6654.0,7907.0,18025.0,43693.0,71472.0


До квантили в 80% счёт идёт на секунды и минуты в действиях, что можно считать действиями внутри одной сессии, далее уже межсессонные действия.

### Чувствительность к значению пробела сессии

In [45]:
gaps = [30, 60, 120, 180, 300, 600, 900, 1800] # 0.5, 1, 2, 3, 5, 10, 15, 30 минут

In [46]:
def session_gap_sensivity(events: pl.DataFrame) -> pl.DataFrame:
    
    rows: list[dict] = []
    for gap in gaps:
        
        # Доля переходов между событиями, при которых начинается новая сессия
        percent_of_breaks: float = (
            events
            .select((pl.col('delta_sec') > gap).mean())
            .item()
        )
        
        # Среднее число сессинй на cookie 
        n_sessions: float = (
            events
            .with_columns(
                (pl.col('delta_sec') > gap)
                .fill_null(True)
                .cum_sum()
                .over('cookie_id')
                .alias('session_id')
            )
            .group_by('cookie_id')
            .agg(pl.col('session_id').n_unique().alias('n_sessions'))
            .select(pl.col('n_sessions').mean())
            .item()
        )
        
        rows.append({
            'gap_sec': gap,
            'percent_of_breaks': f"{round(percent_of_breaks * 100, 2)}%",
            'mean_sessions_per_cookie': round(n_sessions, 2)
        })
        
    return pl.DataFrame(rows)

In [47]:
session_gap_sensivity(events_deltas)

gap_sec,percent_of_breaks,mean_sessions_per_cookie
i64,str,f64
30,"""58.85%""",12.33
60,"""39.29%""",8.56
120,"""24.65%""",5.75
180,"""19.7%""",4.79
300,"""16.5%""",4.18
600,"""14.96%""",3.88
900,"""14.58%""",3.81
1800,"""13.01%""",3.51


In [48]:
def session_gap_by_target(events: pl.DataFrame) -> pl.DataFrame:
    
    # Добавляем target по cookie_id
    df = events.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner')
    
    rows: list[dict] = []
    for gap in gaps:
        
        # Вычисляем количество сессий для людей и ботов
        temp: pl.DataFrame = (
            df
            .with_columns(
                (pl.col('delta_sec') > gap)
                .fill_null(True)
                .cum_sum()
                .over('cookie_id')
                .alias('session_id')
            )
            .group_by(['cookie_id', 'target'])
            .agg(pl.col('session_id').n_unique().alias('n_sessions'))
            .group_by('target')
            .agg(pl.col('n_sessions').mean().alias('mean_sessions'))
        )
        
        human = temp.filter(pl.col('target') == 0).select('mean_sessions').item()
        bot = temp.filter(pl.col('target') == 1).select('mean_sessions').item()
        
        rows.append({
            'gap_sec': gap,
            'mean_sessions_human': round(human, 2),
            'mean_sessions_bot': round(bot, 2),
            'diff_human_bot': round(bot - human, 2)
        })
        
    return pl.DataFrame(rows)

In [49]:
session_gap_target = session_gap_by_target(events_deltas).unpivot(
    index='gap_sec',
    on=['mean_sessions_human', 'mean_sessions_bot', 'diff_human_bot'],
    variable_name='type',
    value_name='mean_sessions',
)

fig = px.bar(
    session_gap_target,
    x='gap_sec',
    y='mean_sessions',
    color='type',
    barmode='group',
    text_auto='.2f'
)

fig.update_xaxes(type='category')

fig.update_layout(height=500, width=1000)

fig.show()

#### Вывод

Чем больше значение gap, тем сложнее становится отделить человека от бота.

Но, если gap ставить слишком маленьким, то количество сессий сильно растёт и может появиться зашумленность.

Хорошие значения gap 180 и 300, а может и среднее между ними.

## Скорость событий

In [50]:
def fast_actions_by_target(events: pl.DataFrame, thresholds: list[float]) -> pl.DataFrame:
    
    df = events.join(train.select(['cookie_id', 'target']), on='cookie_id', how='inner').drop_nulls('delta_sec')
    
    result = (
        df
        .group_by('target')
        .agg(pl.len().alias('n_intervals'))
    )
    
    for t in thresholds:
        temp = (
            df
            .group_by('target')
            .agg(
                (pl.col("delta_sec") < t).sum().alias(f"n_invervals_less_than_{t}s"),
                ((pl.col("delta_sec") < t).mean() * 100).round(2).alias(f"percent_intervals_less_than_{t}s")
            )
        )
        
        result = result.join(temp, on='target', how='left')
    
    return result.sort(by='target')

In [51]:
fast_actions_by_target(events_deltas, thresholds=[0.3, 0.5, 0.7, 1])

target,n_intervals,n_invervals_less_than_0.3s,percent_intervals_less_than_0.3s,n_invervals_less_than_0.5s,percent_intervals_less_than_0.5s,n_invervals_less_than_0.7s,percent_intervals_less_than_0.7s,n_invervals_less_than_1s,percent_intervals_less_than_1s
i64,u32,u32,f64,u32,f64,u32,f64,u32,f64
0,180077,1098,0.61,1098,0.61,1098,0.61,1098,0.61
1,44525,56,0.13,56,0.13,56,0.13,56,0.13


In [52]:
fast_actions_by_target(events_deltas, thresholds=[3, 5, 10, 30, 60])

target,n_intervals,n_invervals_less_than_3s,percent_intervals_less_than_3s,n_invervals_less_than_5s,percent_intervals_less_than_5s,n_invervals_less_than_10s,percent_intervals_less_than_10s,n_invervals_less_than_30s,percent_intervals_less_than_30s,n_invervals_less_than_60s,percent_intervals_less_than_60s
i64,u32,u32,f64,u32,f64,u32,f64,u32,f64,u32,f64
0,180077,4928,2.74,8923,4.96,19920,11.06,61105,33.93,100016,55.54
1,44525,709,1.59,3345,7.51,12924,29.03,27999,62.88,33586,75.43


#### Вывод

У людей доля быстрых действий до 3 секунд немного выше, чем у ботов, но начиная с 5 секунд доля относительно быстрых действий у ботов становится заметно выше, особенно на значениях 10 и 30 секунд.

Стоит добавить признаки на процент интервалов меньше 10 секунд и 30 секунд.